In [0]:
import yaml
from pyspark.sql import functions as F

# Load project config
config_path = "/Workspace/Repos/Mini Projects/Vattenfall-Engineering-Capstone-Project/config/project_config.yml"

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

catalog = config["catalog"]
silver_schema = "refined"
gold_schema = "gold"  

## Cell 2 — Table Names and Logging

We define:

- Fully qualified Silver table names.
- Gold table name.
- A logging function that prints table names, row counts, and the metric being calculated.

This makes notebook runs self-explanatory and easier to debug.

In [0]:
# Silver tables
silver_grid_events_table = f"{catalog}.{silver_schema}.silver_grid_events"
silver_weather_table = f"{catalog}.{silver_schema}.silver_weather"
silver_market_prices_table = f"{catalog}.{silver_schema}.silver_market_prices"

# Gold table
gold_dashboard_table = f"{catalog}.{gold_schema}.gold_regional_operations_dashboard"

# Simple logging function
def log_info(source_table, target_table, rows_read=None, rows_written=None, metric=None):
    print(f"Source Table: {source_table}")
    print(f"Target Table: {target_table}")
    if rows_read is not None:
        print(f"Rows Read: {rows_read}")
    if rows_written is not None:
        print(f"Rows Written: {rows_written}")
    if metric is not None:
        print(f"Business Metric: {metric}")
    print("-" * 50)

##  Load Silver Tables

Load the three Silver tables:

- Grid events
- Weather
- Market prices

We also log the number of rows read from each table.  
This ensures that the inputs are valid before aggregations.

In [0]:
df_events = spark.table(silver_grid_events_table)
df_weather = spark.table(silver_weather_table)
df_market = spark.table(silver_market_prices_table)

# Log row counts
log_info(silver_grid_events_table, gold_dashboard_table, rows_read=df_events.count(), metric="grid events base")
log_info(silver_weather_table, gold_dashboard_table, rows_read=df_weather.count(), metric="weather base")
log_info(silver_market_prices_table, gold_dashboard_table, rows_read=df_market.count(), metric="market prices base")

##  Aggregate Silver Data for Dashboard

We create KPI aggregates for each Silver dataset:

1. **Grid Events**  
   - Total incidents per region per day  
   - Elevated incidents (MODERATE or SEVERE)

2. **Weather**  
   - Compute a binary risk signal (1 if alert level HIGH/SEVERE, else 0)

3. **Market Prices**  
   - Average market price per region per day

Each aggregation is logged to ensure traceability.

In [0]:
# 1️⃣ Grid Events: incident counts by region + day
df_events_agg = df_events.groupBy("region", "event_day").agg(
    F.count("*").alias("incident_count"),
    F.sum(F.when(F.col("severity_band").isin("MODERATE", "SEVERE"), 1).otherwise(0)).alias("elevated_incident_count")
)
log_info(silver_grid_events_table, gold_dashboard_table, rows_written=df_events_agg.count(), metric="aggregated grid incidents")

# 2️⃣ Weather: compute risk indicator
df_weather_agg = df_weather.groupBy("region", "report_day").agg(
    F.max(F.when(F.col("weather_alert_level").isin("HIGH", "SEVERE"), 1).otherwise(0)).alias("weather_risk_signal")
)
log_info(silver_weather_table, gold_dashboard_table, rows_written=df_weather_agg.count(), metric="weather risk signal")

# 3️⃣ Market: average price
df_market_agg = df_market.groupBy("region", "report_day").agg(
    F.avg("price_eur_mwh").alias("avg_market_price")
)
log_info(silver_market_prices_table, gold_dashboard_table, rows_written=df_market_agg.count(), metric="average market price")

##  Combine Aggregates into Dashboard

Join the three aggregates on **region** and **day**:

- Align `event_day` from grid events with `report_day` from weather and market tables
- Select final columns for the dashboard:
    - region
    - report_day
    - incident_count
    - elevated_incident_count
    - weather_risk_signal
    - avg_market_price

Log row counts to ensure joins are successful.

In [0]:
df_dashboard = df_events_agg.join(
    df_weather_agg, 
    (df_events_agg.region == df_weather_agg.region) & (df_events_agg.event_day == df_weather_agg.report_day),
    how="left"
).join(
    df_market_agg,
    (df_events_agg.region == df_market_agg.region) & (df_events_agg.event_day == df_market_agg.report_day),
    how="left"
)

# Select final columns
df_dashboard_final = df_dashboard.select(
    df_events_agg.region,
    df_events_agg.event_day.alias("report_day"),
    "incident_count",
    "elevated_incident_count",
    "weather_risk_signal",
    "avg_market_price"
)

rows_written = df_dashboard_final.count()
log_info("Multiple Silver Tables", gold_dashboard_table, rows_written=rows_written, metric="dashboard-ready KPIs")

##  Save Dashboard Table to Gold

Write the final dashboard table as a **Delta table** in the Gold schema.  

- Mode: overwrite
- Table: `gold_regional_operations_dashboard`
- This table is ready for analyst consumption or dashboards.

In [0]:
df_dashboard_final.write.format("delta").mode("overwrite").saveAsTable(gold_dashboard_table)
print(f"✅ Gold table '{gold_dashboard_table}' created successfully with {rows_written} rows.")

In [0]:
display(df_dashboard_final.limit(5))